In [ ]:
# Required to import modules since it seems cwd is Notebook, rather than where Jupyter was launched from
import sys
sys.path.append('..')

### Setup Data and Strategy

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw_ohlcv/BTCUSDT-1h-2017-08-17.csv", parse_dates=['date'], index_col='date')
df = df[['close', 'volume']].copy()
df = df.loc['2021']

In [ ]:
from src.strategies.ma_crossover import ma_crossover_strategy
import numpy as np

df['position'] = ma_crossover_strategy(df, [50, 100])

# See performance metrics - take multiple
df['return'] = np.log(df['close'].div(df['close'].shift(1)))
df['strategy_returns'] = df['position'].shift(1) * df['return']   # Position shifted since we realise the return for the period after
df['cum_strategy'] = np.exp(df['strategy_returns'].cumsum())

print(f"Multiple: {df['cum_strategy'].iloc[-1]}")

We will use the multiple as a proxy for performance, and optimise the parameters around this metric

### Optimisation

In [ ]:
from itertools import product

In [ ]:
ma_s_range = range(50, 100, 2)

In [ ]:
ma_l_range = range(100, 150, 2)

In [ ]:
combinations = list(product(ma_s_range, ma_l_range))

In [ ]:
total_combinations = len(combinations)
total_combinations

Objective is to try each of these combinations in ma_crossover_strategy to see which one returns the greatest performance (in this case 'Multiple'). For this, I will package up the strategy and performance as a function that takes the paremeters and returns the performance. This way I can loop through each parameter list trying all combinations, and see the results. Upon saving the returned performance, I can view the relative performances

In [ ]:
def strategy_backtest(df, params):
    df['position'] = ma_crossover_strategy(df, params)

    # See performance metrics - take multiple
    df['return'] = np.log(df['close'].div(df['close'].shift(1)))
    df['strategy_returns'] = df['position'].shift(1) * df['return']
    df['cum_strategy'] = np.exp(df['strategy_returns'].cumsum())

    return float(df['cum_strategy'].iloc[-1])

In [ ]:
results = []
for combination in combinations: 
    results.append([combination[0], combination[1], strategy_backtest(df, combination)])
    

In [ ]:
results_df = pd.DataFrame(results)
results_df

### Visualise the Results in Heatmap

In [ ]:
import seaborn as sns

In [ ]:
# 'Pivot' data from a dataframe to a grid by specifying two columns to change to indexes
pivoted_data = pd.pivot(results_df, columns=0, index=1, values=2)
pivoted_data

In [ ]:
sns.heatmap(pivoted_data)